In [ ]:
import pandas as pd
import json
import os
os.chdir('..')

In [16]:
route_nta = pd.read_csv('merge_datasets/route_nta_mapping.csv').dropna(subset=['NTACode'])

route_nta_list = (
    route_nta.groupby('route_id')['NTACode']
    .apply(list)
    .reset_index()
    .rename(columns={'NTACode': 'nta_list'})
)

gtfs_folders = [
    'nta_bus_mapping/data/gtfs_b',
    'nta_bus_mapping/data/gtfs_bx',
    'nta_bus_mapping/data/gtfs_m',
    'nta_bus_mapping/data/gtfs_q',
    'nta_bus_mapping/data/gtfs_si',
]

all_trips  = []
all_shapes = []

for folder in gtfs_folders:
    trips_path  = os.path.join(folder, 'trips.txt')
    shapes_path = os.path.join(folder, 'shapes.txt')
    if os.path.exists(trips_path):
        all_trips.append(pd.read_csv(trips_path, usecols=['route_id','trip_id','shape_id']).drop_duplicates())
    if os.path.exists(shapes_path):
        all_shapes.append(pd.read_csv(shapes_path))

trips_df  = pd.concat(all_trips,  ignore_index=True).drop_duplicates()
shapes_df = pd.concat(all_shapes, ignore_index=True)

route_to_shape = (
    trips_df.sort_values('shape_id')
    .drop_duplicates(subset='route_id', keep='first')
    .set_index('route_id')['shape_id']
    .to_dict()
)

features = []

for _, row in route_nta_list.iterrows():
    route_id = row['route_id']
    nta_list = row['nta_list']

    shape_id = route_to_shape.get(route_id)
    if not shape_id:
        continue

    pts = (
        shapes_df[shapes_df['shape_id'] == shape_id]
        .sort_values('shape_pt_sequence')[['shape_pt_lon','shape_pt_lat']]
        .values.tolist()
    )
    if len(pts) < 2:
        continue

    features.append({
        'type': 'Feature',
        'properties': {
            'route_id': route_id,
            'nta_list': nta_list,
            'tier': 'none',
            'high_need_count': 0
        },
        'geometry': { 'type': 'LineString', 'coordinates': pts }
    })

geojson = {'type': 'FeatureCollection', 'features': features}

with open('nta_map/bus_routes.geojson', 'w') as f:
    json.dump(geojson, f)

print(f"Exported {len(features)} total routes to nta_map/bus_routes.geojson")

Exported 251 total routes to nta_map/bus_routes.geojson
